# SafeVision · Notebook 2x · Does the app catch missing gloves and safety glasses? (experiment E3x)

**Goal:** Notebook 1x measured how well the 8-class detector **finds** gloves, hands, glasses and faces (mAP). That is not the same as the app's real question: *"is this worker missing gloves or safety glasses?"*. Here we test the **whole vision step** (detector + worker matching, `matching.py`) on photos where people marked exactly that.

**Test data:** Ultralytics **Construction-PPE** [C1] (1,000 photos, CC BY 4.0). Besides the PPE boxes, its labels include *missing-PPE* boxes: `no_gloves`, `no_goggle`, `no_helmet`, `no_vest`. Our detector **never trained on it**.

**Fair tuning:** the two confidence thresholds of each item are tuned on the Construction-PPE **train** split (800 photos) and scored **once** on the **val** split (200 photos), the same procedure as Notebook 2 (E3).

| ID | Question | Main scores |
|---|---|---|
| **E3x-a** | Photo level: does the app say "missing gloves / glasses / helmet / vest" when the label says so? | precision, recall, F1 (val) |
| **E3x-b** | Is per-worker matching better than "any bare hand / face in the photo"? | F1: matching vs detector-only |
| **E3x-c** | Is the flagged worker the right one? | share of `no_X` boxes inside a flagged worker |
| **E3x-d** | Does the **zoom re-check** help? (crop each worker with 50 % margin, detect again at 640 px, merge) | F1: matching vs matching + zoom, chosen per item on train |

Two views of the negatives: **all** photos without the missing-PPE label, and **clear** photos where the item is visibly worn (a `gloves` / `goggles` box exists). The clear view avoids counting photos where hands or eyes are simply not visible.

**Runtime:** GPU, about 10 min.

## 0. Install

In [ ]:
# [colab-only]
!nvidia-smi
%pip install -q ultralytics openpyxl huggingface_hub pyyaml

## 1. Configuration (edit this cell only)

In [ ]:
# CONFIG
from pathlib import Path
import re, json, random, time, os, importlib.util

ON_KAGGLE = Path("/kaggle/input").exists()
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BASE = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if IN_COLAB else Path.home() / "safevision"
OUT = BASE / "safevision_outputs_nb2x"
ART = BASE / "artifacts"
DETECTOR_WEIGHTS = ART / "detector_ppe8.pt"
CPPE_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
CPPE_RAW = BASE / "raw_cppe"

IMGSZ = 960                     # overwritten by run_config.json of Notebook 1x
DET_CONF = 0.03                 # very weak boxes are kept in the cache; the app itself uses boxes >= 0.10
ZOOM_PAD, ZOOM_IMGSZ = 0.5, 640 # zoom re-check: crop margin around each worker and detection size of the crop
MATCH_METHOD = "box"
GRID_YES = [0.25, 0.40, 0.55]
GRID_NO = [0.25, 0.35, 0.45, 0.55, 0.65]
SEED = 42

# item -> (missing-PPE label in Construction-PPE, worn-PPE label, bare body-part class of our detector)
ITEMS = {"gloves": ("no_gloves", "gloves", "hands"), "glasses": ("no_goggle", "goggles", "face"),
         "helmet": ("no_helmet", "helmet", "head"), "vest": ("no_vest", "vest", None)}

RES, FIGS = OUT / "results", OUT / "figures"
for p in [OUT, RES, FIGS, ART, CPPE_RAW]:
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
HF_REPO, token = None, None

def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

## 2. Results logger

In [ ]:
import csv, datetime
import pandas as pd
LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "method", "item", "dataset", "split", "subset", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, method, item, split, subset, metric, value, notes=""):
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, method=method, item=item, dataset="Ultralytics Construction-PPE",
                        split=split, subset=subset, metric=metric, value=round(float(value), 4), notes=notes,
                        timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

## 3. Get the 8-class detector from Notebook 1x
From your HF repo (`nb1x_ppe8_detector`, or the finished checkpoint `checkpoints/E1x_yolo11s_960_ppe8/best_final.pt`), a Kaggle input, or an upload.

In [ ]:
# [colab-only]
import shutil, zipfile, io
from huggingface_hub import login, HfApi, hf_hub_download
token = get_secret("HF_TOKEN")
if token:
    login(token=token)
    HF_REPO = f"{HfApi(token=token).whoami()['name']}/safevision-artifacts"
    for f, dest in [("nb1x_ppe8_detector/models/detector_best.pt", DETECTOR_WEIGHTS),
                    ("checkpoints/E1x_yolo11s_960_ppe8/best_final.pt", DETECTOR_WEIGHTS),
                    ("nb1x_ppe8_detector/run_config.json", ART / "run_config_ppe8.json")]:
        if dest.exists():
            continue
        try:
            shutil.copy(hf_hub_download(HF_REPO, f, token=token), dest); print("got", f)
        except Exception:
            pass
if not DETECTOR_WEIGHTS.exists() and ON_KAGGLE:
    hits = [p for p in Path("/kaggle/input").rglob("*.pt") if "ppe8" in str(p).lower() or "e1x" in str(p).lower()]
    if hits:
        shutil.copy(hits[0], DETECTOR_WEIGHTS)
while not DETECTOR_WEIGHTS.exists() and IN_COLAB:
    from google.colab import files
    print("Upload the 8-class detector (detector_best.pt from safevision_nb1x_outputs.zip):")
    up = files.upload()
    if not up:
        break
    DETECTOR_WEIGHTS.write_bytes(list(up.values())[0])
assert DETECTOR_WEIGHTS.exists(), "8-class detector not found"

In [ ]:
from ultralytics import YOLO
if (ART / "run_config_ppe8.json").exists():
    IMGSZ = json.load(open(ART / "run_config_ppe8.json")).get("best_imgsz", IMGSZ)
detector = YOLO(str(DETECTOR_WEIGHTS))
NAMES = detector.names if isinstance(detector.names, dict) else dict(enumerate(detector.names))
print("classes:", list(NAMES.values()), "| imgsz", IMGSZ)
assert {"hands", "gloves", "face", "glasses"} <= set(NAMES.values()), "This is not the 8-class detector"

## 4. Worker matching (same file as the web-app backend: `backend/app/matching.py`)

In [ ]:
# ---- backend/app/matching.py ----
"""SafeVision worker matching (step V2).

Turns raw detector boxes into one record per worker with a status for each PPE item:
"yes", "no" or "unsure" ("n/a" if the detector cannot see that item).
  * 4-class detector (Notebook 1):  person, head, helmet, vest
  * 8-class detector (Notebook 1x): + hands, gloves, face, glasses
Each item is checked on its own body part: helmet on the head, vest on the torso,
gloves on the hands, safety glasses on the face.

Two ways to find where a worker's head and torso are:
  * "box":  fixed fractions of the person box (top 30% = head area,
            20-65% = torso area). Simple, no extra model.
  * "pose": body keypoints from YOLO11-pose (nose/eyes/ears = head,
            shoulders/hips = torso), following the idea of matching PPE
            to body parts with pose estimation (Lopez et al., 2025, paper [4]).
If pose keypoints are missing for a person, "pose" falls back to "box".

The same file is used by Notebook 2 (experiments E3/E4) and by the backend,
so the app behaves exactly like the evaluated system.
"""
from __future__ import annotations

CLASSES = ["person", "head", "helmet", "vest", "hands", "gloves", "face", "glasses"]
# PPE item -> (bare body-part class that proves it is missing, PPE class that proves it is worn)
EXTRA_ITEMS = {"gloves": ("hands", "gloves"), "glasses": ("face", "glasses")}

# COCO keypoint indices used by YOLO11-pose
HEAD_KPTS = [0, 1, 2, 3, 4]          # nose, left/right eye, left/right ear
TORSO_KPTS = [5, 6, 11, 12]          # left/right shoulder, left/right hip


# ----------------------------------------------------------------- geometry
def area(b):
    """Area of an [x1, y1, x2, y2] box (0 if the box is empty)."""
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def intersection(a, b):
    """Area where two boxes overlap."""
    return area([max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])])


def iou(a, b):
    """Intersection over union of two boxes (0 = no overlap, 1 = identical)."""
    inter = intersection(a, b)
    union = area(a) + area(b) - inter
    return inter / union if union > 0 else 0.0


def inside_fraction(inner, outer):
    """Share of `inner` that lies inside `outer` (e.g. how much of a helmet is in the head area)."""
    a = area(inner)
    return intersection(inner, outer) / a if a > 0 else 0.0


# ------------------------------------------------------------ detections
def detections_from_result(result, min_conf=0.10):
    """Convert one Ultralytics detection result into {class_name: [(box, conf), ...]}."""
    names = result.names
    known = set(names.values()) if isinstance(names, dict) else set(names)
    dets = {c: [] for c in CLASSES if c in known or c in ("person", "head", "helmet", "vest")}   # missing classes -> "n/a"
    for box, conf, cls in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.tolist()):
        name = names[int(cls)]
        if name in dets and conf >= min_conf:
            dets[name].append((box, float(conf)))
    return dets


# ------------------------------------------------------- zoom re-check
ZOOM_ITEMS = {"helmet": ("head", "helmet"), "vest": ("vest",), "gloves": ("hands", "gloves"), "glasses": ("face", "glasses")}


def zoom_detections(model, image, person_boxes, imgsz=640, conf=0.03, pad=0.15, items=("helmet", "vest", "gloves")):
    """Zoom re-check: crop every detected worker (box grown by `pad`), run the detector again on the enlarged crop
    and return the PPE / body-part boxes in full-image coordinates. Small helmets, gloves and faces become large
    enough to detect (the idea of SAHI slicing, Akyon et al. 2022, focused on each person). Only the classes of
    `items` are returned (Notebook 2x decides which items profit)."""
    keep = {c for it in items for c in ZOOM_ITEMS[it]}
    W, H = image.size
    crops, offsets = [], []
    for b in person_boxes:
        x1, y1, x2, y2 = expand(b, pad)
        x1, y1, x2, y2 = max(0, int(x1)), max(0, int(y1)), min(W, int(x2)), min(H, int(y2))
        if x2 - x1 > 4 and y2 - y1 > 4:
            crops.append(image.crop((x1, y1, x2, y2))); offsets.append((x1, y1))
    out = []
    if not crops:
        return out
    for (ox, oy), r in zip(offsets, model.predict(crops, imgsz=imgsz, conf=conf, verbose=False)):
        for b, s, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()):
            name = r.names[int(c)]
            if name in keep:
                out.append((name, [b[0] + ox, b[1] + oy, b[2] + ox, b[3] + oy], float(s)))
    return out


def merge_detections(dets, extra, min_conf=0.10, same_iou=0.5):
    """Add zoomed boxes to a {class: [(box, conf)]} dict. A zoomed box that overlaps a box of the same class
    (IoU > same_iou) is the same object: only the more confident of the two is kept."""
    for name, b, s in extra:
        if s < min_conf or name not in dets:
            continue
        same = [i for i, (ob, oc) in enumerate(dets[name]) if iou(ob, b) > same_iou]
        if not same:
            dets[name].append((b, s))
        else:
            i = max(same, key=lambda k: dets[name][k][1])
            if s > dets[name][i][1]:
                dets[name][i] = (b, s)
    return dets


def poses_from_result(result, kpt_conf=0.5):
    """Convert one YOLO11-pose result into a list of (person_box, {kpt_index: (x, y)})."""
    out = []
    if result.keypoints is None:
        return out
    xy = result.keypoints.xy.tolist()
    kc = result.keypoints.conf.tolist() if result.keypoints.conf is not None else None
    for i, box in enumerate(result.boxes.xyxy.tolist()):
        pts = {}
        for k, (x, y) in enumerate(xy[i]):
            if kc is None or kc[i][k] >= kpt_conf:
                pts[k] = (x, y)
        out.append((box, pts))
    return out


# ------------------------------------------------------- body regions
def box_regions(p):
    """Head and torso areas as fixed fractions of a person box (method 'box')."""
    x1, y1, x2, y2 = p
    h = y2 - y1
    head = [x1, y1 - 0.05 * h, x2, y1 + 0.30 * h]      # a little above the box too: helmets stick out
    torso = [x1, y1 + 0.20 * h, x2, y1 + 0.65 * h]
    return head, torso


def pose_regions(p, pose_list):
    """Head and torso areas from the best-matching pose skeleton (method 'pose'); None if unavailable."""
    best, best_iou = None, 0.5                          # pose person must overlap the detected person well
    for pbox, pts in pose_list:
        v = iou(p, pbox)
        if v > best_iou:
            best, best_iou = pts, v
    if best is None:
        return None
    head_pts = [best[k] for k in HEAD_KPTS if k in best]
    torso_pts = [best[k] for k in TORSO_KPTS if k in best]
    if len(head_pts) < 2 or len(torso_pts) < 2:
        return None
    ph = p[3] - p[1]
    hx = [x for x, _ in head_pts]; hy = [y for _, y in head_pts]
    cx, cy = sum(hx) / len(hx), sum(hy) / len(hy)
    s = max(max(hx) - min(hx), max(hy) - min(hy), 0.08 * ph)
    head = [cx - 1.2 * s, cy - 2.0 * s, cx + 1.2 * s, cy + 0.8 * s]     # helmets sit above the eyes
    tx = [x for x, _ in torso_pts]; ty = [y for _, y in torso_pts]
    pad = 0.1 * (max(tx) - min(tx) + 1)
    torso = [min(tx) - pad, min(ty) - pad, max(tx) + pad, max(ty) + pad]
    return head, torso


def expand(b, f):
    """Box grown by a fraction f of its size on every side (hands stick out of the person box)."""
    w, h = b[2] - b[0], b[3] - b[1]
    return [b[0] - f * w, b[1] - f * h, b[2] + f * w, b[3] + f * h]


def covered(b, items, thr=0.3):
    """True if box b overlaps one of the boxes in items (e.g. a 'hands' box that is really a gloved hand)."""
    return any(iou(b, o) > thr or inside_fraction(b, o) > 0.6 for o, _ in items)


def _item_status(dets, item, region, used, conf_yes, conf_no):
    """Status of an extra PPE item (gloves on hands, glasses on face) inside one body region."""
    bare_cls, ppe_cls = EXTRA_ITEMS[item]
    if bare_cls not in dets or ppe_cls not in dets:
        return "n/a", 0.0
    ppe = _best(dets[ppe_cls], region, 0.5, used.setdefault(ppe_cls, set()))
    if ppe and ppe[2] >= conf_yes:
        used[ppe_cls].add(ppe[0])
        return "yes", ppe[2]
    bare = [(i, b, c) for i, (b, c) in enumerate(dets[bare_cls])
            if i not in used.setdefault(bare_cls, set()) and inside_fraction(b, region) >= 0.5 and c >= conf_no
            and not covered(b, dets[ppe_cls])]
    if bare:
        i, b, c = max(bare, key=lambda t: t[2]); used[bare_cls].add(i)
        return "no", c
    return "unsure", ppe[2] if ppe else 0.0


def _best(items, region, min_inside, used):
    """Highest-confidence item (index, box, conf) mostly inside `region` and not used by another worker."""
    best = None
    for i, (b, c) in enumerate(items):
        if i in used:
            continue
        if inside_fraction(b, region) >= min_inside and (best is None or c > best[2]):
            best = (i, b, c)
    return best


# ------------------------------------------------------------ main function
def match_workers(dets, img_h, pose_list=None, method="box", conf_yes=0.40, conf_no=0.40,
                  min_person_conf=0.30, item_conf=None):
    """Assign helmets, heads and vests to workers.

    Returns a list of worker dicts:
      box, source ('person' or 'head-only'), person_conf,
      helmet ('yes'/'no'/'unsure'), helmet_conf, vest ('yes'/'no'/'unsure'), vest_conf.

    Rules (helmet):  matched helmet with conf >= conf_yes            -> 'yes'
                     else matched bare head with conf >= conf_no      -> 'no'
                     else                                             -> 'unsure'
    Gloves / glasses (8-class detector only):
                     PPE box (gloves / glasses) on the hand / face area, conf >= conf_yes -> 'yes'
                     else a bare hand / face there (not overlapping a PPE box), conf >= conf_no -> 'no'
                     else -> 'unsure' (hands or face not visible); 'n/a' with the 4-class detector
    Rules (vest):    matched vest with conf >= conf_yes               -> 'yes'
                     else person clearly visible (conf >= 0.5 and
                          at least 15% of the image height)           -> 'no'
                     else                                             -> 'unsure'
    item_conf: optional {"gloves": (conf_yes, conf_no), "glasses": (...)} to tune those items separately
    (tuned in Notebook 2x, experiment E3x); otherwise they use conf_yes / conf_no.
    Bare heads that belong to no detected person (occlusion, crowds) become
    extra 'head-only' workers, so a hidden body does not hide a missing helmet.
    """
    workers = []
    item_conf = item_conf or {}
    gy, gn = item_conf.get("gloves", (conf_yes, conf_no))
    ey, en = item_conf.get("glasses", (conf_yes, conf_no))
    used = {"head": set(), "helmet": set(), "vest": set()}
    persons = sorted([p for p in dets["person"] if p[1] >= min_person_conf], key=lambda t: -t[1])
    for pbox, pconf in persons:
        regions = pose_regions(pbox, pose_list) if (method == "pose" and pose_list) else None
        head_r, torso_r = regions if regions else box_regions(pbox)
        helmet = _best(dets["helmet"], head_r, 0.5, used["helmet"])
        head = _best(dets["head"], head_r, 0.5, used["head"])
        vest = _best(dets["vest"], torso_r, 0.3, used["vest"])
        for key, hit in (("helmet", helmet), ("head", head), ("vest", vest)):
            if hit:
                used[key].add(hit[0])
        if helmet and helmet[2] >= conf_yes:
            h_status, h_conf = "yes", helmet[2]
        elif head and head[2] >= conf_no:
            h_status, h_conf = "no", head[2]
        else:
            h_status, h_conf = "unsure", max([x[2] for x in (helmet, head) if x] or [0.0])
        visible = pconf >= 0.5 and (pbox[3] - pbox[1]) >= 0.15 * img_h
        if vest and vest[2] >= conf_yes:
            v_status, v_conf = "yes", vest[2]
        elif visible:
            v_status, v_conf = "no", pconf
        else:
            v_status, v_conf = "unsure", vest[2] if vest else 0.0
        g_status, g_conf = _item_status(dets, "gloves", expand(pbox, 0.15), used, gy, gn)
        e_status, e_conf = _item_status(dets, "glasses", head_r, used, ey, en)
        workers.append(dict(box=pbox, source="person", person_conf=pconf, helmet=h_status,
                            helmet_conf=h_conf, vest=v_status, vest_conf=v_conf,
                            gloves=g_status, gloves_conf=g_conf, glasses=e_status, glasses_conf=e_conf))
    # Bare heads with no person around them -> extra workers without a helmet.
    for i, (hb, hc) in enumerate(dets["head"]):
        if i in used["head"] or hc < conf_no:
            continue
        if any(iou(hb, hl) > 0.3 or inside_fraction(hl, hb) > 0.5 for hl, _ in dets["helmet"]):
            continue                                       # a helmet sits on this head
        workers.append(dict(box=hb, source="head-only", person_conf=0.0, helmet="no",
                            helmet_conf=hc, vest="unsure", vest_conf=0.0,
                            gloves="unsure" if "gloves" in dets else "n/a", gloves_conf=0.0,
                            glasses=_item_status(dets, "glasses", expand(hb, 0.1), used, ey, en)[0]
                            if "glasses" in dets else "n/a", glasses_conf=0.0))
    return workers


def image_has_helmet_violation(workers):
    """Image-level decision: at least one worker clearly without a helmet."""
    return any(w["helmet"] == "no" for w in workers)

## 5. Download Construction-PPE and read the labels
Every photo keeps its split (train / val) and its labelled boxes by class name.

In [ ]:
import urllib.request, zipfile, yaml
from PIL import Image
if not any(CPPE_RAW.rglob("*.jpg")):
    urllib.request.urlretrieve(CPPE_URL, CPPE_RAW / "cppe.zip")
    zipfile.ZipFile(CPPE_RAW / "cppe.zip").extractall(CPPE_RAW)

def cppe_names():
    import ultralytics
    for y in list(CPPE_RAW.rglob("*.yaml")) + [Path(ultralytics.__file__).parent / "cfg/datasets/construction-ppe.yaml"]:
        try:
            n = yaml.safe_load(open(y)).get("names")
            if n:
                return [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
        except Exception:
            pass
    return ["helmet", "gloves", "vest", "boots", "goggles", "none", "Person", "no_helmet", "no_goggle", "no_gloves", "no_boots", "no_vest"]

CNAMES = [n.lower() for n in cppe_names()]
print("Construction-PPE classes:", CNAMES)
records = []
for lp in sorted(CPPE_RAW.rglob("*.txt")):
    parts = lp.parts
    if "labels" not in parts:
        continue
    split = "val" if any(p in ("val", "valid", "test") for p in parts) else "train"
    imgs = [p for ext in (".jpg", ".jpeg", ".png") for p in [Path(str(lp).replace("labels", "images")).with_suffix(ext)] if p.exists()]
    if not imgs:
        continue
    W, H = Image.open(imgs[0]).size
    boxes = []
    for line in lp.read_text(errors="ignore").splitlines():
        v = line.split()
        if len(v) == 5 and int(v[0]) < len(CNAMES):
            c, x, y, w, h = int(v[0]), *map(float, v[1:])
            boxes.append((CNAMES[c], [(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H]))
    records.append(dict(image=str(imgs[0]), split=split, W=W, H=H, boxes=boxes))
print(pd.Series([r["split"] for r in records]).value_counts().to_dict(), "photos")

## 6. Run the detector once on every photo
Boxes are cached in `detections.json`, so threshold tuning is instant and a restarted run skips this step.

In [ ]:
DET_CACHE = RES / "detections.json"
dets_cache = json.load(open(DET_CACHE)) if DET_CACHE.exists() else {}
todo = [r for r in records if r["image"] not in dets_cache]
for k, rec in enumerate(todo):
    im = Image.open(rec["image"]).convert("RGB")
    res = detector.predict(im, imgsz=IMGSZ, conf=DET_CONF, verbose=False)[0]
    full = [(NAMES[int(c)], b, float(s)) for b, s, c in zip(res.boxes.xyxy.tolist(), res.boxes.conf.tolist(), res.boxes.cls.tolist())]
    persons = [b for n, b, s in full if n == "person" and s >= 0.30]
    zoom = zoom_detections(detector, im, persons, imgsz=ZOOM_IMGSZ, conf=DET_CONF, pad=ZOOM_PAD,
                           items=("helmet", "vest", "gloves", "glasses"))       # all items; E3x-d decides which ones profit
    dets_cache[rec["image"]] = {"full": full, "zoom": zoom}
    if (k + 1) % 100 == 0:
        print(f"{k + 1}/{len(todo)} photos"); json.dump(dets_cache, open(DET_CACHE, "w"))
json.dump(dets_cache, open(DET_CACHE, "w"))
print("photos with detections:", len(dets_cache))

def dets_for(path, zoom_items=()):
    # Detections of one photo as {class: [(box, conf)]}, boxes >= 0.10 as in the app; plus zoomed boxes of zoom_items.
    d = {c: [] for c in CLASSES}
    for name, b, s in dets_cache[path]["full"]:
        if name in d and s >= 0.10:
            d[name].append((b, s))
    if zoom_items:
        keep = {c for it in zoom_items for c in ZOOM_ITEMS[it]}
        d = merge_detections(d, [x for x in dets_cache[path]["zoom"] if x[0] in keep])
    return d

## 7. Ground truth and the two methods

- **Positive photo** for an item: it has at least one missing-PPE box (e.g. `no_gloves`).
- **Matching (the app):** `match_workers` with that item's thresholds → photo is flagged if any worker gets `"no"`.
- **Detector-only (baseline):** photo is flagged if any bare body part (`hands` / `face` / `head`) is detected anywhere with conf ≥ threshold and is not covered by a PPE box. No worker logic. (Not defined for the vest, which has no bare-body-part class.)

In [ ]:
import numpy as np

def truth(rec, item):
    # 1 = missing PPE labelled, 0 = not; 'clear' = the item is visibly worn in the photo (a worn-PPE box exists)
    miss, worn, _ = ITEMS[item]
    names = [n for n, _ in rec["boxes"]]
    return int(miss in names), (miss not in names and worn in names)

def app_flags(rec, item, cy, cn, zoom=False):
    d = dets_for(rec["image"], (item,) if zoom else ())
    if item in ("gloves", "glasses"):
        w = match_workers(d, rec["H"], method=MATCH_METHOD, conf_yes=0.40, conf_no=0.40, item_conf={item: (cy, cn)})
    else:
        w = match_workers(d, rec["H"], method=MATCH_METHOD, conf_yes=cy, conf_no=cn)
    return w

def predict(rec, item, cy, cn, method):
    if method == "detector-only":
        _, worn_cls, bare = ITEMS[item]
        ppe = {"gloves": "gloves", "glasses": "glasses", "helmet": "helmet"}[item]
        d = dets_for(rec["image"])
        return int(any(c >= cn and not covered(b, d[ppe]) for b, c in d[bare]))
    return int(any(w.get(item) == "no" for w in app_flags(rec, item, cy, cn, zoom=method == "matching+zoom")))

def scores(y, p):
    y, p = np.array(y), np.array(p)
    tp, fp, fn = int(((y == 1) & (p == 1)).sum()), int(((y == 0) & (p == 1)).sum()), int(((y == 1) & (p == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0; rec_ = tp / (tp + fn) if tp + fn else 0.0
    return dict(precision=prec, recall=rec_, F1=2 * prec * rec_ / (prec + rec_) if prec + rec_ else 0.0,
                accuracy=float((y == p).mean()) if len(y) else 0.0, positives=int((y == 1).sum()), negatives=int((y == 0).sum()))

def evaluate(split, item, cy, cn, method, subset="all"):
    ys, ps = [], []
    for r in records:
        if r["split"] != split:
            continue
        y, clear = truth(r, item)
        if subset == "clear" and not (y == 1 or clear):
            continue
        ys.append(y); ps.append(predict(r, item, cy, cn, method))
    return scores(ys, ps)

## 8. Tune on train, score once on val (E3x-a, E3x-b)

In [ ]:
rows, best, zoom_choice = [], {}, {}
for item in ITEMS:
    train_f1 = {}
    for method in (["matching", "matching+zoom", "detector-only"] if ITEMS[item][2] else ["matching", "matching+zoom"]):
        grid = [(cy, cn) for cy in GRID_YES for cn in GRID_NO] if method != "detector-only" else [(0, cn) for cn in GRID_NO]
        tuned = max(grid, key=lambda t: evaluate("train", item, *t, method, "clear")["F1"])
        train_f1[method] = (evaluate("train", item, *tuned, method, "clear")["F1"], tuned)
        for subset in ("all", "clear"):
            s = evaluate("val", item, *tuned, method, subset)
            rows.append(dict(item=item, method=method, subset=subset, conf_yes=tuned[0], conf_no=tuned[1], **s))
            for metric in ("precision", "recall", "F1", "accuracy", "positives", "negatives"):
                log_result("E3x", f"E3x_{item}_{method}", method, item, "val", subset, metric, s[metric],
                           f"thresholds tuned on train: yes={tuned[0]}, no={tuned[1]}")
    # E3x-d: the app uses zoom for this item only if it was better on the TRAIN split
    use_zoom = train_f1["matching+zoom"][0] > train_f1["matching"][0]
    best[item] = list(train_f1["matching+zoom" if use_zoom else "matching"][1]); zoom_choice[item] = use_zoom
table = pd.DataFrame(rows).round(3)
print(table.to_string(index=False))
table.to_csv(RES / "E3x_scores.csv", index=False)
best["zoom_items"] = [k for k, v in zoom_choice.items() if v]
json.dump(best, open(RES / "matching_thresholds_ppe8.json", "w"), indent=2)
print("tuned settings (copy to backend/models):", best)

## 9. E3x-c · Is the flagged worker the right one?
For each labelled missing-PPE box in the val photos: does its centre fall inside a worker (box grown by 15 %) that the app flagged `"no"` for that item?

In [ ]:
loc = []
for item in ITEMS:
    miss = ITEMS[item][0]; hits = total = 0
    for r in records:
        if r["split"] != "val":
            continue
        gt = [b for n, b in r["boxes"] if n == miss]
        if not gt:
            continue
        flagged = [expand(w["box"], 0.15) for w in app_flags(r, item, *best[item], zoom=zoom_choice[item]) if w.get(item) == "no"]
        for b in gt:
            cx, cy_ = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
            total += 1; hits += any(f[0] <= cx <= f[2] and f[1] <= cy_ <= f[3] for f in flagged)
    rate = hits / total if total else 0.0
    loc.append(dict(item=item, labelled_boxes=total, inside_flagged_worker=hits, rate=round(rate, 3)))
    log_result("E3x", f"E3x_loc_{item}", "matching", item, "val", "all", "localisation_rate", rate, f"{total} labelled boxes")
loc = pd.DataFrame(loc); print(loc.to_string(index=False)); loc.to_csv(RES / "E3x_localisation.csv", index=False)

## 10. Chart, examples and saving

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw
v = table[table.subset == "clear"].pivot(index="item", columns="method", values="F1")
ax = v.plot.bar(figsize=(8, 4), rot=0, color=["#94a3b8", "#2563eb", "#16a34a"][: v.shape[1]])
ax.set_title("E3x · Photo-level F1 on Construction-PPE val (clear subset)"); ax.set_ylim(0, 1); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(FIGS / "E3x_F1.png", dpi=150); plt.show()

COL = {"yes": (22, 163, 74), "no": (220, 38, 38), "unsure": (234, 179, 8)}
ex = [r for r in records if r["split"] == "val" and any(n in ("no_gloves", "no_goggle") for n, _ in r["boxes"])][:4]
fig, axes = plt.subplots(1, max(1, len(ex)), figsize=(5 * max(1, len(ex)), 5), squeeze=False)
for ax, r in zip(axes[0], ex):
    im = Image.open(r["image"]).convert("RGB"); dr = ImageDraw.Draw(im)
    ws = match_workers(dets_for(r["image"], best["zoom_items"]), r["H"], item_conf={k: tuple(best[k]) for k in ("gloves", "glasses")})
    for w in ws:
        bad = [k for k in ("helmet", "vest", "gloves", "glasses") if w.get(k) == "no"]
        dr.rectangle(w["box"], outline=COL["no"] if bad else COL["yes"], width=4)
        dr.text((w["box"][0] + 4, w["box"][1] + 4), "no " + ",".join(bad) if bad else "ok", fill=(255, 255, 255))
    ax.imshow(im); ax.axis("off"); ax.set_title(", ".join(sorted({n for n, _ in r["boxes"] if n.startswith("no_")})), fontsize=9)
plt.tight_layout(); plt.savefig(FIGS / "E3x_examples.png", dpi=120); plt.show()

In [ ]:
import shutil
with pd.ExcelWriter(RES / "E3x_ppe8_matching.xlsx") as xw:
    read_log().to_excel(xw, "all_results", index=False)
    table.to_excel(xw, "E3x_scores", index=False)
    loc.to_excel(xw, "E3x_localisation", index=False)
archive = shutil.make_archive(str(OUT.parent / "safevision_nb2x_outputs"), "zip", OUT)
print("Saved", archive)

In [ ]:
# [colab-only]
if HF_REPO:
    HfApi(token=token).upload_folder(folder_path=str(OUT), repo_id=HF_REPO, path_in_repo="nb2x_ppe8_matching",
                                     ignore_patterns=["*detections.json"])
    print("Uploaded to", HF_REPO, "/ nb2x_ppe8_matching")
if IN_COLAB:
    from google.colab import files
    files.download(archive)

## References
- [C1] Ultralytics, "Construction-PPE dataset," docs.ultralytics.com/datasets/detect/construction-ppe, 2026 (CC BY 4.0).
- [4] Lopez et al. 2025: matching PPE to body parts (basis of `matching.py`, Notebook 2).